# Make a dataset from a jsonl file

In [ ]:
import pandas as pd    
jsonObj = pd.read_json(path_or_buf="../records_export_full/stift-heiligenkreuz__2026-08-05T06-46-32Z.jsonl", lines=True)

# get only the necessary columns
df = pd.DataFrame(jsonObj)
del df["kp_id"]
del df["src_id"]
del df["institution_id"]
df


In [ ]:
mydict = df["content"].to_list()
mydict[:5]

In [ ]:
from collections import Counter

def find_key(data, key):
    """Yield every value stored under `key`, at any depth."""
    if isinstance(data, dict):
        for k, v in data.items():
            if k == key:
                yield v
            else:
                yield from find_key(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_key(item, key)

def unpack(v):
    if isinstance(v, list):
        for item in v:
            yield from unpack(item)
    elif isinstance(v, dict):
        if "@value" in v:
            yield v["@value"], v.get("@language")
    elif v is not None:
        yield v, None

value_counts = Counter()
value_langs = {}

for d in mydict:
    for medium in find_key(d, "medium"):
        for value, lang in unpack(medium):
            value_counts[value] += 1
            value_langs[value] = lang

print(value_counts)
print(value_langs)

In [ ]:
from collections import Counter

def find_values(data, key=None):
    if isinstance(data, dict):
        if "@value" in data and key is None:
            yield data
        for k, v in data.items():
            if key is not None and k == key:
                yield from find_values(v)
            else:
                yield from find_values(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_values(item, key)


value_counts = Counter()
value_langs = {}

for d in mydict:
    for obj in find_values(d, key="medium"):
        value = obj["@value"]
        lang = obj.get("@language")   # not every @value has a language
        value_counts[value] += 1
        value_langs[value] = lang

print(dict(value_counts))
print(value_langs)

In [ ]:
data = {"word": value_counts.keys(),
        "count": value_counts.values(),
        #"lang": value_langs.values(),
        }
df = pd.DataFrame(data)
df = df.sort_values("count", ascending=False)
df

In [ ]:
df.loc[df['lang'] == "de", 'word']

In [ ]:
en_list = df.loc[df['lang'] == "de", 'word'].to_list()

In [ ]:
stift_list = df["word"].to_list()
print(len(stift_list))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(stift_list)

clustering = AgglomerativeClustering(
    distance_threshold=0.8,  
    n_clusters=None,
    linkage='ward'
)

clustering.fit(embeddings)

# Step 4: Group words by cluster labels
labels = clustering.labels_
clusters = {}
for word, label in zip(stift_list, labels):
    clusters.setdefault(label, []).append(word)

# Step 5: Print clusters
for label, cluster_words in clusters.items():
    print(f"Cluster {label}: {cluster_words}")
print(len(clusters))